# B.3 Investigation: SISC with the Authors' Simulated Time Series Data
## Replication of Appendix B.3 from FTS-Diffusion (ICLR 2024)

This notebook replicates the **Appendix B.3** investigation of the FTS-Diffusion paper using the **exact simulated data the authors provide** in `fts-diffusion-ref/data/data_toy_l10-20_*.csv`.

### Goals
1. Load the **authors' simulated time series** (10,000 points, K=4 patterns) from the repo
2. Run **SISC** on this data
3. Compute **per-unit DTW error** and **Jaccard similarity (IoU)**
4. Compare to paper-reported results:
   - **Multi-pattern (4 patterns)**: avg per-unit DTW = **0.01**, Jaccard = **0.784**

### Why Use Their Data?
The authors include the simulated data in their repo (`data_toy_l10-20_*.csv`):
- 10,000 data points (matches paper)
- 668 segments with **ground-truth pattern labels** (K=4)
- **Ground-truth segmentation boundaries** (lengths 10-20)

This means we have the **exact ground truth** they used. No guessing pattern shapes.

## Setup: Clone Repo on Google Colab

In [ ]:
# Detect environment and set repo path
import os, sys, subprocess

IS_COLAB = 'google.colab' in sys.modules
print(f"Running on Colab: {IS_COLAB}")

if IS_COLAB:
    # Clone the repo on Colab
    REPO_URL = 'https://github.com/thaidinger/ml-in-fcs.git'
    BRANCH = 'deli_work'
    CLONE_DIR = '/content/ml-in-fcs'
    
    if not os.path.exists(CLONE_DIR):
        print(f"Cloning {REPO_URL} (branch={BRANCH})...")
        subprocess.check_call(['git', 'clone', '-b', BRANCH, REPO_URL, CLONE_DIR])
    else:
        print(f"Repo already exists at {CLONE_DIR}, pulling latest...")
        subprocess.check_call(['git', '-C', CLONE_DIR, 'pull'])
    
    REPO_ROOT = f'{CLONE_DIR}/fts-diffusion-ref'
else:
    REPO_ROOT = '/home/deli/ETH_projects/ml-in-fcs/fts-diffusion-ref'

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print(f"Working dir: {os.getcwd()}")
print(f"Files in data/: {os.listdir('data')}")

In [ ]:
# Install dependencies
for pkg in ['numpy', 'pandas', 'matplotlib', 'scipy', 'scikit-learn', 'torch', 'tqdm', 'tslearn', 'dtaidistance']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print("\u2713 Dependencies ready")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment
import warnings
warnings.filterwarnings('ignore')

# SISC and DTW from authors' code
from models.pattern_recognition_module import SISC
from tslearn.metrics import dtw as tslearn_dtw

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("\u2713 Imports ready")

## 1. Load the Authors' Simulated Data

Three files in `data/`:
- `data_toy_l10-20_timeseries.csv`: the 10,000-point time series
- `data_toy_l10-20_labels.csv`: ground-truth pattern label per segment (K=4)
- `data_toy_l10-20_segmentation.csv`: ground-truth segment boundaries

In [ ]:
DATA_DIR = 'data'

# Load files
ts = pd.read_csv(f'{DATA_DIR}/data_toy_l10-20_timeseries.csv', index_col=0).values.flatten().astype(np.float32)
real_labels = pd.read_csv(f'{DATA_DIR}/data_toy_l10-20_labels.csv', index_col=0).values.flatten().astype(int)
real_boundaries = pd.read_csv(f'{DATA_DIR}/data_toy_l10-20_segmentation.csv', index_col=0).values.flatten().astype(int)

K_TRUE = len(np.unique(real_labels))
L_MIN, L_MAX = 10, 20  # From file name

print(f"Time series length:    {len(ts)}")
print(f"Time series range:     [{ts.min():.4f}, {ts.max():.4f}]")
print(f"Number of segments:    {len(real_labels)}")
print(f"Number of boundaries:  {len(real_boundaries)}")
print(f"Unique patterns (K):   {K_TRUE}")
print(f"Pattern frequency:     {np.bincount(real_labels)}")
print(f"Segment lengths:       min={np.diff(real_boundaries).min()}, max={np.diff(real_boundaries).max()}, mean={np.diff(real_boundaries).mean():.2f}")

In [ ]:
# Visualize: full series + zoom-in showing segment boundaries colored by pattern
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# Full time series
axes[0].plot(ts, linewidth=0.5, color='black')
axes[0].set_title('Authors\' Simulated Time Series (full, 10,000 points)')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Value')
axes[0].grid(alpha=0.3)

# Zoom-in with colored segments
ZOOM_END = 300
colors = ['steelblue', 'coral', 'green', 'purple']
axes[1].plot(ts[:ZOOM_END], linewidth=1.0, color='black', alpha=0.5)
for i, label in enumerate(real_labels):
    if real_boundaries[i+1] > ZOOM_END:
        break
    seg_start = real_boundaries[i]
    seg_end = real_boundaries[i+1]
    axes[1].axvspan(seg_start, seg_end, alpha=0.2, color=colors[label])
    axes[1].plot(range(seg_start, seg_end), ts[seg_start:seg_end], linewidth=1.5, color=colors[label])

axes[1].set_title(f'Zoom-in (first {ZOOM_END} points): colors = ground-truth pattern labels')
axes[1].set_xlabel('Time')
axes[1].grid(alpha=0.3)
# Custom legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[k], alpha=0.4, label=f'Pattern {k}') for k in range(K_TRUE)]
axes[1].legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

## 2. Compute Ground-Truth Centroids

The paper defines the ground-truth centroid as the **DBA barycenter of all segments belonging to the same pattern**. We extract those segments and compute the centroid per pattern.

In [ ]:
from tslearn.barycenters import dtw_barycenter_averaging

# Extract real segments per pattern
real_segments_per_pattern = {k: [] for k in range(K_TRUE)}
for i, label in enumerate(real_labels):
    seg = ts[real_boundaries[i]:real_boundaries[i+1]]
    real_segments_per_pattern[label].append(seg)

for k in range(K_TRUE):
    print(f"Pattern {k}: {len(real_segments_per_pattern[k])} segments")

# Compute DBA barycenter per pattern (this is the ground-truth centroid)
L_REF = L_MAX  # Use l_max as reference length for centroid
ground_truth_centroids = []

print("\nComputing ground-truth DBA barycenters...")
for k in range(K_TRUE):
    # Convert to list of arrays for tslearn (each shape (l, 1))
    segs_2d = [s.reshape(-1, 1) for s in real_segments_per_pattern[k]]
    barycenter = dtw_barycenter_averaging(segs_2d, max_iter=20).flatten()
    # Resample to L_REF for consistent comparison
    if len(barycenter) != L_REF:
        x_old = np.linspace(0, 1, len(barycenter))
        x_new = np.linspace(0, 1, L_REF)
        barycenter = np.interp(x_new, x_old, barycenter)
    ground_truth_centroids.append(barycenter)
    print(f"  Pattern {k}: centroid range [{barycenter.min():.4f}, {barycenter.max():.4f}]")

# Visualize ground-truth centroids
fig, axes = plt.subplots(1, K_TRUE, figsize=(4*K_TRUE, 3))
for k in range(K_TRUE):
    axes[k].plot(ground_truth_centroids[k], linewidth=2, color=colors[k])
    axes[k].set_title(f'Ground-truth centroid p{k+1}')
    axes[k].grid(alpha=0.3)
fig.suptitle('Ground-Truth Centroids (DBA Barycenters)', fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 3. Run SISC on the Authors' Simulated Data

Configuration matches the paper:
- `n_clusters=4` (K)
- `l_min=10, l_max=20`
- `init_strategy='kmeans++'`
- `barycenter='dba'`
- `max_iters=10` (paper uses up to 20)

In [ ]:
print("Running SISC on authors' simulated data (this takes 5-10 min)...")

sisc = SISC(n_clusters=K_TRUE, l_min=L_MIN, l_max=L_MAX)
sisc.fit(
    ts,
    max_iters=10,
    init_strategy='kmeans++',
    barycenter='dba',
    plot_progress=False,
    store_res=False)

print(f"\n\u2713 SISC completed.")
print(f"  Number of centroids: {len(sisc.centroids)}")
print(f"  Number of segments learned: {len(sisc.subsequences)}")
print(f"  Final loss: {sisc.total_loss:.4f}")

## 4. Evaluation Metrics

### 4.1 Per-unit DTW Error
From the paper:
$$\text{per-unit DTW} = \frac{\text{DTW}(c_{\text{real}}, c_{\text{sisc}})}{L}$$
Computed in **normalized space** (each centroid rescaled to $[0,1]$).

In [ ]:
def normalize_to_unit(arr):
    """Min-max normalize a 1D array to [0, 1]."""
    arr = np.asarray(arr, dtype=np.float64)
    mn, mx = arr.min(), arr.max()
    if mx - mn < 1e-12:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

def per_unit_dtw_error(c_real, c_sisc):
    """Per-unit DTW error in normalized space (paper's definition)."""
    c_real_n = normalize_to_unit(c_real)
    c_sisc_n = normalize_to_unit(c_sisc)
    L = len(c_real_n)
    return tslearn_dtw(c_real_n, c_sisc_n) / L

# Get SISC centroids
sisc_centroids = [np.array(c) for c in sisc.centroids]

# Hungarian alignment: match each SISC cluster to the closest ground-truth
dist_matrix = np.zeros((K_TRUE, K_TRUE))
for i in range(K_TRUE):
    for j in range(K_TRUE):
        dist_matrix[i, j] = per_unit_dtw_error(ground_truth_centroids[i], sisc_centroids[j])

row_ind, col_ind = linear_sum_assignment(dist_matrix)
alignment = dict(zip(row_ind, col_ind))  # ground-truth pattern -> SISC cluster
print(f"Alignment (ground-truth \u2192 SISC cluster): {alignment}")

# Per-pattern DTW error
errors_per_pattern = []
for gt_idx, sisc_idx in alignment.items():
    err = per_unit_dtw_error(ground_truth_centroids[gt_idx], sisc_centroids[sisc_idx])
    errors_per_pattern.append(err)
    print(f"  Pattern {gt_idx} \u2192 SISC {sisc_idx}: per-unit DTW = {err:.4f}")

avg_dtw_error = np.mean(errors_per_pattern)
print(f"\nAverage Per-unit DTW Error: {avg_dtw_error:.4f}")
print(f"Paper reported:             0.01")

### 4.2 Jaccard Similarity (Segmentation IoU)

From the paper:
$$\text{IoU} = \frac{|s_{\text{real}} \cap s_{\text{sisc}}|}{|s_{\text{real}} \cup s_{\text{sisc}}|}$$

Operating on the boundaries (segmentation points).

In [ ]:
def jaccard_segmentation(boundaries_real, boundaries_sisc, tol=2):
    """
    Jaccard similarity between two sets of segmentation boundaries.
    A real boundary 'matches' a SISC boundary if they're within tol points.
    """
    matched_real = 0
    for b in boundaries_real:
        if any(abs(b - bs) <= tol for bs in boundaries_sisc):
            matched_real += 1
    
    intersection = matched_real
    union = len(boundaries_real) + len(boundaries_sisc) - intersection
    return intersection / union if union > 0 else 0.0


def jaccard_pointwise(real_labels, real_boundaries, sisc_labels, sisc_boundaries, total_length, alignment):
    """
    Pointwise Jaccard: per-point label agreement after Hungarian alignment.
    Build dense per-point label arrays from boundaries+labels.
    """
    real_dense = np.full(total_length, -1, dtype=int)
    for i in range(len(real_labels)):
        if i + 1 < len(real_boundaries):
            real_dense[real_boundaries[i]:real_boundaries[i+1]] = real_labels[i]
    
    sisc_dense = np.full(total_length, -1, dtype=int)
    for i in range(len(sisc_labels)):
        if i + 1 < len(sisc_boundaries):
            sisc_dense[sisc_boundaries[i]:sisc_boundaries[i+1]] = sisc_labels[i]
    
    # Align SISC labels via Hungarian mapping
    inv_alignment = {sisc_k: gt_k for gt_k, sisc_k in alignment.items()}
    sisc_aligned = np.array([inv_alignment.get(s, -1) if s >= 0 else -1 for s in sisc_dense])
    
    valid = (real_dense >= 0) & (sisc_aligned >= 0)
    if valid.sum() == 0:
        return 0.0
    return (real_dense[valid] == sisc_aligned[valid]).mean()

# Boundary-based IoU
sisc_boundaries = np.array(sisc.segmentation)
iou_boundaries = jaccard_segmentation(real_boundaries, sisc_boundaries, tol=2)
print(f"Jaccard (boundary-based, tol=2): {iou_boundaries:.4f}")

# Point-based IoU
sisc_labels_arr = np.array(sisc.labels)
iou_pointwise = jaccard_pointwise(
    real_labels, real_boundaries,
    sisc_labels_arr, sisc_boundaries,
    len(ts), alignment)
print(f"Jaccard (pointwise, after alignment): {iou_pointwise:.4f}")

print(f"\nPaper reported: 0.784")

## 5. Visualize Results: Replicate Fig. 9

Side-by-side comparison of standard patterns, ground-truth centroids, and SISC-learned centroids.

In [ ]:
fig, axes = plt.subplots(K_TRUE, 3, figsize=(13, 3*K_TRUE))

for i in range(K_TRUE):
    sisc_idx = alignment[i]
    
    # Column 0: Series excerpt where this pattern occurs
    seg_indices = np.where(real_labels == i)[0]
    if len(seg_indices) > 0:
        seg_start = real_boundaries[seg_indices[0]]
        seg_end = real_boundaries[seg_indices[0] + 1]
        ctx_start = max(0, seg_start - 20)
        ctx_end = min(len(ts), seg_end + 20)
        axes[i, 0].plot(range(ctx_start, ctx_end), ts[ctx_start:ctx_end],
                       linewidth=0.8, color='gray', alpha=0.5)
        axes[i, 0].plot(range(seg_start, seg_end), ts[seg_start:seg_end],
                       linewidth=2, color=colors[i])
        axes[i, 0].axvspan(seg_start, seg_end, alpha=0.2, color=colors[i])
    axes[i, 0].set_title(f'(a) Series excerpt: pattern {i}')
    axes[i, 0].grid(alpha=0.3)
    
    # Column 1: Ground-truth centroid
    axes[i, 1].plot(normalize_to_unit(ground_truth_centroids[i]),
                   linewidth=2, color='green')
    axes[i, 1].set_title(f'(c) Ground-truth p{i+1} (normalized)')
    axes[i, 1].set_ylim([-0.05, 1.05])
    axes[i, 1].grid(alpha=0.3)
    
    # Column 2: SISC-learned centroid
    axes[i, 2].plot(normalize_to_unit(sisc_centroids[sisc_idx]),
                   linewidth=2, color='coral')
    axes[i, 2].set_title(f'(d) SISC p{i+1} (DTW={errors_per_pattern[i]:.4f})')
    axes[i, 2].set_ylim([-0.05, 1.05])
    axes[i, 2].grid(alpha=0.3)

fig.suptitle(
    f'Fig. 9 Replication: Multi-pattern SISC Investigation\n'
    f'Avg per-unit DTW error = {avg_dtw_error:.4f} (paper: 0.01) | '
    f'IoU (boundary) = {iou_boundaries:.4f}, (pointwise) = {iou_pointwise:.4f} (paper: 0.784)',
    fontsize=13, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

## 6. Confusion Matrix: Pattern Assignment Accuracy

How often does SISC assign segments to the same pattern as the ground truth?

In [ ]:
# To compare per-segment, we need to align the two segmentations.
# For each ground-truth segment, find the SISC label of the majority of its points.

sisc_labels_arr = np.array(sisc.labels)
sisc_dense = np.full(len(ts), -1, dtype=int)
for i in range(len(sisc_labels_arr)):
    if i + 1 < len(sisc_boundaries):
        sisc_dense[sisc_boundaries[i]:sisc_boundaries[i+1]] = sisc_labels_arr[i]

# Map SISC labels to ground-truth labels via alignment
inv_alignment = {sisc_k: gt_k for gt_k, sisc_k in alignment.items()}
sisc_dense_aligned = np.array([inv_alignment.get(s, -1) for s in sisc_dense])

# Per-segment majority vote
from collections import Counter
predicted_labels_per_real_segment = []
for i in range(len(real_labels)):
    seg_start = real_boundaries[i]
    seg_end = real_boundaries[i+1]
    sisc_in_segment = sisc_dense_aligned[seg_start:seg_end]
    sisc_in_segment = sisc_in_segment[sisc_in_segment >= 0]
    if len(sisc_in_segment) > 0:
        majority = Counter(sisc_in_segment).most_common(1)[0][0]
        predicted_labels_per_real_segment.append(majority)
    else:
        predicted_labels_per_real_segment.append(-1)

predicted_labels_per_real_segment = np.array(predicted_labels_per_real_segment)

# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
valid = predicted_labels_per_real_segment >= 0
cm = confusion_matrix(real_labels[valid], predicted_labels_per_real_segment[valid])

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues', interpolation='nearest')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
               color='white' if cm[i, j] > cm.max()/2 else 'black')
ax.set_xlabel('Predicted (SISC, aligned)')
ax.set_ylabel('Ground-truth')
ax.set_title('Pattern Assignment Confusion Matrix')
ax.set_xticks(range(K_TRUE))
ax.set_yticks(range(K_TRUE))
plt.colorbar(im)
plt.tight_layout()
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(real_labels[valid], predicted_labels_per_real_segment[valid],
                            target_names=[f'Pattern {k}' for k in range(K_TRUE)]))

segment_accuracy = (real_labels[valid] == predicted_labels_per_real_segment[valid]).mean()
print(f"Per-segment accuracy: {segment_accuracy:.4f}")

## 7. Final Results Summary

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Per-unit DTW (Pattern 0)',
        'Per-unit DTW (Pattern 1)',
        'Per-unit DTW (Pattern 2)',
        'Per-unit DTW (Pattern 3)',
        'Avg Per-unit DTW',
        'Jaccard (boundary)',
        'Jaccard (pointwise)',
        'Per-segment accuracy',
    ],
    'Paper': [
        '~0.01', '~0.01', '~0.01', '~0.01',
        '0.01',
        '0.784',
        '-',
        '-',
    ],
    'Ours': [
        f'{errors_per_pattern[0]:.4f}',
        f'{errors_per_pattern[1]:.4f}',
        f'{errors_per_pattern[2]:.4f}',
        f'{errors_per_pattern[3]:.4f}',
        f'{avg_dtw_error:.4f}',
        f'{iou_boundaries:.4f}',
        f'{iou_pointwise:.4f}',
        f'{segment_accuracy:.4f}',
    ],
})

print("\n" + "="*60)
print("B.3 Investigation Results: Authors' Toy Dataset")
print("="*60)
print(summary.to_string(index=False))
print("="*60)

## 8. One-Pattern Scenario (Fig. 8): Derived from the Toy Data

The authors don't ship a one-pattern toy file, so we **derive it** from the existing 4-pattern toy data:
1. Pick one pattern $p \in \{0, 1, 2, 3\}$ (we'll test all 4).
2. Extract all segments labeled $p$ from the original 10k series.
3. Concatenate them with continuity adjustment (offset each segment so its first point equals the previous segment's last point) — same trick as the authors' sampling pipeline.
4. Run SISC with $K=1$ on the resulting series.
5. Compare with the paper-reported one-pattern numbers: **per-unit DTW = 0.009, Jaccard = 0.938**.

This gives a fair one-pattern test using the **authors' actual segments** as building blocks.

In [ ]:
def build_one_pattern_series(ts, real_labels, real_boundaries, target_pattern):
    """
    Build a single-pattern time series by concatenating all segments of `target_pattern`
    with continuity adjustment.

    Returns:
        new_ts: 1D array of the reconstructed series
        new_segments: list of arrays, each segment's values
        new_boundaries: cumulative segment boundaries (length = #segments + 1)
        original_segment_indices: indices into the original real_labels
    """
    # Find segments of the target pattern
    seg_indices = np.where(real_labels == target_pattern)[0]
    
    new_segments_raw = []
    for i in seg_indices:
        segment = ts[real_boundaries[i]:real_boundaries[i+1]]
        new_segments_raw.append(segment)

    # Concatenate with continuity adjustment (offset alignment)
    parts = [new_segments_raw[0]]
    for seg in new_segments_raw[1:]:
        offset = parts[-1][-1] - seg[0]
        parts.append(seg + offset)
    new_ts = np.concatenate(parts).astype(np.float32)

    # Boundaries of the new series
    lengths = [len(s) for s in new_segments_raw]
    new_boundaries = np.concatenate([[0], np.cumsum(lengths)])
    
    return new_ts, parts, new_boundaries, seg_indices

# Build one-pattern series for pattern 0 as a demo
TARGET_PATTERN_DEMO = 0
ts_one, segs_one, bdry_one, orig_idx_one = build_one_pattern_series(
    ts, real_labels, real_boundaries, TARGET_PATTERN_DEMO)

print(f"Pattern {TARGET_PATTERN_DEMO}: {len(segs_one)} segments")
print(f"Reconstructed series length: {len(ts_one)}")
print(f"Segment lengths: min={np.diff(bdry_one).min()}, max={np.diff(bdry_one).max()}, mean={np.diff(bdry_one).mean():.2f}")

# Visualize: the new one-pattern series + a zoom-in
fig, axes = plt.subplots(2, 1, figsize=(14, 5))
axes[0].plot(ts_one, linewidth=0.5, color='black')
axes[0].set_title(f'One-pattern series (pattern {TARGET_PATTERN_DEMO}): full reconstruction')
axes[0].grid(alpha=0.3)

ZOOM = min(300, len(ts_one))
axes[1].plot(ts_one[:ZOOM], linewidth=1.0, color='black', alpha=0.5)
for i in range(min(15, len(segs_one))):
    if bdry_one[i+1] > ZOOM:
        break
    axes[1].axvspan(bdry_one[i], bdry_one[i+1], alpha=0.2, color=colors[TARGET_PATTERN_DEMO])
axes[1].set_title(f'Zoom-in (first {ZOOM} points), shaded = segment boundaries')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 8.1 Run SISC with K=1 and Compute Metrics

We'll loop over all 4 patterns to see how SISC performs on each as a one-pattern problem.

In [ ]:
def evaluate_one_pattern(ts_one, segs_one, bdry_one, target_pattern, ground_truth_centroid):
    """Run SISC with K=1 on a one-pattern series and compute metrics."""
    sisc_op = SISC(n_clusters=1, l_min=L_MIN, l_max=L_MAX)
    sisc_op.fit(
        ts_one,
        max_iters=10,
        init_strategy='kmeans++',
        barycenter='dba',
        plot_progress=False,
        store_res=False)

    sisc_op_centroid = np.array(sisc_op.centroids[0])
    sisc_op_boundaries = np.array(sisc_op.segmentation)

    dtw_err = per_unit_dtw_error(ground_truth_centroid, sisc_op_centroid)
    iou_b = jaccard_segmentation(bdry_one, sisc_op_boundaries, tol=2)

    return {
        'pattern': target_pattern,
        'series_length': len(ts_one),
        'n_segments_real': len(segs_one),
        'n_segments_sisc': len(sisc_op.subsequences),
        'sisc_centroid': sisc_op_centroid,
        'dtw_error': dtw_err,
        'iou_boundary': iou_b,
        'sisc_boundaries': sisc_op_boundaries,
    }


one_pattern_results = {}
for k in range(K_TRUE):
    print(f"\n=== Pattern {k} ===")
    ts_k, segs_k, bdry_k, _ = build_one_pattern_series(ts, real_labels, real_boundaries, k)
    print(f"  Series length: {len(ts_k)}, segments: {len(segs_k)}")
    
    print(f"  Running SISC (K=1)...")
    res = evaluate_one_pattern(ts_k, segs_k, bdry_k, k, ground_truth_centroids[k])
    res['ts'] = ts_k
    res['real_boundaries'] = bdry_k
    one_pattern_results[k] = res
    
    print(f"  Per-unit DTW error: {res['dtw_error']:.4f}  (paper: 0.009)")
    print(f"  Jaccard (boundary): {res['iou_boundary']:.4f}  (paper: 0.938)")

### 8.2 Replicate Fig. 8 for Each Pattern

Side-by-side: simulated series excerpt | ground-truth centroid | SISC-learned centroid.

In [ ]:
fig, axes = plt.subplots(K_TRUE, 4, figsize=(16, 3*K_TRUE))

for i in range(K_TRUE):
    res = one_pattern_results[i]
    ts_i = res['ts']
    bdry_i = res['real_boundaries']

    # (a) Simulated series excerpt
    ZOOM = min(200, len(ts_i))
    axes[i, 0].plot(ts_i[:ZOOM], linewidth=0.8, color='black', alpha=0.6)
    for j in range(min(12, len(bdry_i)-1)):
        if bdry_i[j+1] > ZOOM:
            break
        axes[i, 0].axvspan(bdry_i[j], bdry_i[j+1], alpha=0.15, color=colors[i])
    axes[i, 0].set_title(f'(a) One-pattern series p{i+1}')
    axes[i, 0].grid(alpha=0.3)

    # (b) Ground-truth centroid (normalized)
    axes[i, 1].plot(normalize_to_unit(ground_truth_centroids[i]),
                   linewidth=2, color='green')
    axes[i, 1].set_title(f'(c) Ground-truth p{i+1} (norm)')
    axes[i, 1].set_ylim([-0.05, 1.05])
    axes[i, 1].grid(alpha=0.3)

    # (c) SISC-learned centroid (normalized)
    axes[i, 2].plot(normalize_to_unit(res['sisc_centroid']),
                   linewidth=2, color='coral')
    axes[i, 2].set_title(f"(d) SISC p{i+1} (DTW={res['dtw_error']:.4f})")
    axes[i, 2].set_ylim([-0.05, 1.05])
    axes[i, 2].grid(alpha=0.3)

    # (d) Overlay: ground-truth vs SISC (normalized) for direct comparison
    axes[i, 3].plot(normalize_to_unit(ground_truth_centroids[i]),
                   linewidth=2, color='green', label='Ground-truth')
    axes[i, 3].plot(normalize_to_unit(res['sisc_centroid']),
                   linewidth=2, color='coral', linestyle='--', label='SISC')
    axes[i, 3].set_title(f'(e) Overlay p{i+1}')
    axes[i, 3].set_ylim([-0.05, 1.05])
    axes[i, 3].legend(fontsize=8)
    axes[i, 3].grid(alpha=0.3)

fig.suptitle(
    'Fig. 8 Replication: One-Pattern SISC Investigation (per pattern)',
    fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

### 8.3 One-Pattern Summary Table

In [ ]:
one_pattern_summary = pd.DataFrame([
    {
        'Pattern': k,
        'Series length': one_pattern_results[k]['series_length'],
        '# Segments (real)': one_pattern_results[k]['n_segments_real'],
        '# Segments (SISC)': one_pattern_results[k]['n_segments_sisc'],
        'Per-unit DTW': f"{one_pattern_results[k]['dtw_error']:.4f}",
        'Jaccard (boundary)': f"{one_pattern_results[k]['iou_boundary']:.4f}",
    }
    for k in range(K_TRUE)
])

# Add a paper-reported row for comparison
paper_row = pd.DataFrame([{
    'Pattern': 'Paper (Fig. 8)',
    'Series length': 10000,
    '# Segments (real)': '~700',
    '# Segments (SISC)': '~700',
    'Per-unit DTW': '0.009',
    'Jaccard (boundary)': '0.938',
}])

combined = pd.concat([one_pattern_summary, paper_row], ignore_index=True)
print("\n" + "="*80)
print("One-Pattern (Fig. 8) Replication Results")
print("="*80)
print(combined.to_string(index=False))
print("="*80)

# Aggregated stats across the 4 derived one-pattern series
avg_dtw_op = np.mean([r['dtw_error'] for r in one_pattern_results.values()])
avg_iou_op = np.mean([r['iou_boundary'] for r in one_pattern_results.values()])
print(f"\nAverage across 4 patterns:")
print(f"  Avg per-unit DTW: {avg_dtw_op:.4f}  (paper: 0.009)")
print(f"  Avg Jaccard:      {avg_iou_op:.4f}  (paper: 0.938)")

### Caveats for the One-Pattern Replication

Our derived one-pattern series differs from the paper's Fig. 8 in a few ways:

1. **Series length is shorter**: ~2,500 points (one pattern's 169 segments) vs. 10,000 in the paper. SISC has fewer segments to learn from, which may slightly hurt centroid quality.
2. **Segments come from a multi-pattern series**: each segment was originally in a context where it followed a different-pattern segment, so the absolute values may have a small drift bias.
3. **Pattern shape differs by pattern**: the paper's Fig. 8 likely uses a specific 'mountain' pattern; our derived data uses each of the 4 toy patterns in turn — none may match Fig. 8 exactly.

To match the paper exactly, the authors would need to publish their one-pattern toy file or the generation script. The above is the closest we can get with what they shipped.

## 9. Discussion

### What This Replication Shows
- We use the **exact toy dataset the authors include in their repo** (`data_toy_l10-20_*.csv`).
- This is the multi-pattern (K=4) scenario from Fig. 9 of the paper.
- The simulated series has length 10,000 with 668 segments and segment lengths 10-20, matching the paper.

### How to Interpret the Results
- **Per-unit DTW error**: should be ~0.01 (paper). Higher ⇒ SISC centroids drift from ground truth.
- **Jaccard (boundary)**: more strict; counts boundary alignment within ±2 points.
- **Jaccard (pointwise)**: looser; counts per-point label agreement after alignment.
- **Confusion matrix**: shows which patterns SISC confuses most often.

### If Results Differ from the Paper
Possible reasons:
1. **Random initialization**: SISC uses kmeans++ which has stochasticity. Try different seeds.
2. **`max_iters`**: Paper uses up to 20 iterations; we use 10 here. Try increasing.
3. **Jaccard definition**: Paper doesn't specify the exact tolerance for boundary matching.
4. **DBA convergence**: barycenter computation can have local minima.

### Connection to Predictable Drift
If SISC's centroids deviate from ground truth (high DTW error), the downstream pattern generator and evolution module are trained on imprecise pattern definitions. This is one mechanism that could introduce **artifactual predictability** into synthetic time series.

### Next Steps
- Run with different random seeds to assess variance in SISC results.
- Increase `max_iters` to 20 (paper setting).
- Test with K mismatched (e.g., K=3 or K=5) to see how SISC degrades.
- Add noise robustness sweep on top of the toy data.